# 04: Trend tests — did the outsourcing penalty change over time?

Specification: the author's model (`refs/author_spec.md`), saved in `outputs/py/estimates.csv` as `spec = author`.
Every test uses that spec.

Part A tests the by-period estimates already in `outputs/py/estimates.csv` (no refitting). Period samples use disjoint
survey months, and SEs are clustered by survey month, so estimates from different periods are treated as independent.
Trend SEs are shown two ways: unscaled (known-variance WLS) and scaled by sqrt(max(1, Q_resid / df_resid)), per
`refs/author_spec.md`. Contrasts use the unscaled SE, as the z-formula specifies.

Part B refits a pooled 2001–2025 model with period and linear-year interactions (`outsourced` interacted with period dummies
or with year, covariate coefficients held constant across periods). Part B results can differ from Part A.

Single years 1999–2005 are shown as a plot (CLAUDE.md open issue: check the guard penalty around the 2003 code change).

Outputs: `outputs/py/trend_tests.csv`, `outputs/figures/single_year_1999_2005.png`, `outputs/RESULTS_trends.md`.

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import pyfixest as pf

OUT = "../outputs"
est = pd.read_csv(f"{OUT}/py/estimates.csv")
plt.rcParams["font.family"] = ["Times New Roman", "Times", "serif"]

BINS = ["1983-1985", "1986-1988", "1989-1991", "1992-1994", "1995-1997", "1998-2000",
        "2001-2003", "2004-2006", "2007-2009", "2010-2012", "2013-2015", "2016-2019", "2021-2025"]
EXT_BINS = [b for b in BINS if int(b[:4]) >= 2001]
OCC_LABEL = {"janitor": "Janitors", "guard": "Guards"}

def midpoint(label):
    y0, y1 = (int(v) for v in label.split("-"))
    return (y0 + y1) / 2

def by_period(occ):
    sub = est[(est["spec"] == "author") & (est["model"] == "base") & (est["term"] == "outsourced")
              & (est["occ"] == occ)].set_index("sample")
    return sub.loc[BINS, ["estimate", "se"]]

# 5-year bins, 2001-2025 only (same bins as Tables 1, 2, 3, and 4): robustness check on Part A's
# bin width, alongside the 3-year/13-bin BINS/EXT_BINS above. 2016-2019 is a shorter 4-year bin;
# 2020 is excluded from every bin.
BINS5 = ["2001-2005", "2006-2010", "2011-2015", "2016-2019", "2021-2025"]

def by_period5(occ):
    sub = est[(est["spec"] == "author") & (est["model"] == "base") & (est["term"] == "outsourced")
              & (est["occ"] == occ)].set_index("sample")
    return sub.loc[BINS5, ["estimate", "se"]]

## Part A: by-period tests

Homogeneity: inverse-variance weighted mean `b_bar`, `Q = sum(w_k (b_k - b_bar)^2)`, `w_k = 1/se_k^2`, chi-square with K-1 df.
Linear trend: inverse-variance weighted regression of `b_k` on the period midpoint year. Reported per decade. The unscaled SE
is the known-variance WLS SE. The scaled SE multiplies it by sqrt(max(1, Q_resid / df_resid)), with df_resid = K - 2.
Contrasts: `z = (b1 - b2) / sqrt(se1^2 + se2^2)`. Convergence: guard minus janitor, SE `sqrt(se_guard^2 + se_janitor^2)`.

In [ ]:
def homogeneity(b, se):
    w = 1 / se**2
    bbar = np.sum(w * b) / np.sum(w)
    Q = np.sum(w * (b - bbar) ** 2)
    df = len(b) - 1
    return dict(estimate=bbar, se=np.sqrt(1 / np.sum(w)), statistic=Q, df=df, p=stats.chi2.sf(Q, df))

def wls_trend(b, se, x):
    w = 1 / se**2
    X = np.column_stack([np.ones_like(x), x])
    XtWX = X.T @ (X * w[:, None])
    cov = np.linalg.inv(XtWX)
    beta = np.linalg.solve(XtWX, X.T @ (w * b))
    slope, se_slope = beta[1], np.sqrt(cov[1, 1])
    Q_resid = np.sum(w * (b - X @ beta) ** 2)
    scale = np.sqrt(max(1.0, Q_resid / (len(b) - 2)))
    z = slope / se_slope
    z_s = slope / (se_slope * scale)
    return dict(estimate=10 * slope, se=10 * se_slope, statistic=z, df=np.nan, p=2 * stats.norm.sf(abs(z)),
                se_scaled=10 * se_slope * scale, p_scaled=2 * stats.norm.sf(abs(z_s)))

def contrast(b1, se1, b2, se2):
    d = b1 - b2
    s = math.sqrt(se1**2 + se2**2)
    z = d / s
    return dict(estimate=d, se=s, statistic=z, df=np.nan, p=2 * stats.norm.sf(abs(z)))

rows_a = []
def add(occupation, sample, test, res):
    rows_a.append(dict(occupation=occupation, sample=sample, test=test, **res))

for occ in ["janitor", "guard"]:
    P = by_period(occ)
    for samp, labs in [("1983-2025", BINS), ("2001-2025", EXT_BINS)]:
        b = P.loc[labs, "estimate"].to_numpy(dtype=float)
        se = P.loc[labs, "se"].to_numpy(dtype=float)
        x = np.array([midpoint(l) for l in labs])
        add(OCC_LABEL[occ], samp, "Homogeneity: penalty constant across periods (Q)", homogeneity(b, se))
        add(OCC_LABEL[occ], samp, "Linear trend per decade (inverse-variance WLS)", wls_trend(b, se, x))

# 5-year bins, 2001-2025 (robustness on Part A's bin width; Part B's model-based linear trend
# remains the primary trend test -- see refs/author_spec.md, "Trend tests")
for occ in ["janitor", "guard"]:
    P5 = by_period5(occ)
    b = P5["estimate"].to_numpy(dtype=float)
    se = P5["se"].to_numpy(dtype=float)
    x = np.array([midpoint(l) for l in BINS5])
    add(OCC_LABEL[occ], "2001-2025 (5-year bins)", "Homogeneity: penalty constant across periods (Q)", homogeneity(b, se))
    add(OCC_LABEL[occ], "2001-2025 (5-year bins)", "Linear trend per decade (inverse-variance WLS)", wls_trend(b, se, x))

CONTRASTS = [
    ("1983-1985", "2021-2025", "1983-2025", "Contrast: 1983-1985 minus 2021-2025 (long-run change)", ["janitor", "guard"]),
    ("1998-2000", "2001-2003", "1983-2025", "Contrast: 1998-2000 minus 2001-2003 (replication/extension boundary)", ["janitor", "guard"]),
    ("2001-2003", "2021-2025", "2001-2025", "Contrast: 2001-2003 minus 2021-2025 (change within extension)", ["janitor", "guard"]),
    ("2013-2015", "2021-2025", "2001-2025", "Contrast: 2013-2015 minus 2021-2025 (guards only)", ["guard"]),
]
for p1, p2, samp, name, occs in CONTRASTS:
    for occ in occs:
        P = by_period(occ)
        add(OCC_LABEL[occ], samp, name,
            contrast(P.loc[p1, "estimate"], P.loc[p1, "se"], P.loc[p2, "estimate"], P.loc[p2, "se"]))

G, J = by_period("guard"), by_period("janitor")
gap = G["estimate"] - J["estimate"]
gap_se = np.sqrt(G["se"] ** 2 + J["se"] ** 2)
for samp, labs in [("1983-2025", BINS), ("2001-2025", EXT_BINS)]:
    b = gap.loc[labs].to_numpy(dtype=float)
    se = gap_se.loc[labs].to_numpy(dtype=float)
    x = np.array([midpoint(l) for l in labs])
    add("Guard minus janitor", samp, "Convergence: homogeneity of guard-minus-janitor gap (Q)", homogeneity(b, se))
    add("Guard minus janitor", samp, "Convergence: linear trend per decade in guard-minus-janitor gap", wls_trend(b, se, x))

part_a = pd.DataFrame(rows_a)
print(part_a.round(4).to_string(index=False))

## Part B: pooled 2001–2025 model with period and trend interactions

Primary trend test for the body (per the Paper structure). Period interactions use the 5-year bins (2001-2005 .. 2021-2025, matching Table 3/Figure 2; reference 2001-2005); the 3-year/13-bin version (reference 2001-2003) is kept as an appendix robustness row, sample `"2001-2025 (3-year bins)"`. The linear outsourced x year trend model is unchanged -- same formula and sample as before, independent of which period-interaction scheme is used.

In [ ]:
author_formula = (
    "log_wage ~ outsourced + union + parttime + educ_cat + female + black + hispanic "
    "+ age + age2 + msa + central_city + cc_unknown + geo_unidentified | STATEFIP^YEAR"
)
cols_needed = ["occupation", "YEAR", "outsourced", "EARNWT", "ym", "STATEFIP", "extension_sample",
               "log_wage", "union", "parttime", "educ_cat", "female", "black", "hispanic", "age", "age2",
               "msa", "central_city", "cc_unknown", "geo_unidentified"]
dk_pool = pd.read_parquet(f"{OUT}/py/dk_clean.parquet", columns=cols_needed)
ext_pool = dk_pool[dk_pool["extension_sample"]].copy()

# Primary: 5-year bins (matches Table 3 / Figure 2). Reference: 2001-2005.
REF5 = "2001-2005"
year_to_label5 = {y: l for l in BINS5 for y in range(int(l[:4]), int(l[5:]) + 1)}
ext_pool["period5"] = ext_pool["YEAR"].map(year_to_label5)
interact_labels5 = [b for b in BINS5 if b != REF5]
interact_cols5 = {b: "op5_" + b.replace("-", "_") for b in interact_labels5}
for b, c in interact_cols5.items():
    ext_pool[c] = ext_pool["outsourced"] * (ext_pool["period5"] == b)

# Appendix robustness: 3-year/13-bin scheme (as previously used). Reference: 2001-2003.
REF3 = "2001-2003"
year_to_label3 = {y: l for l in EXT_BINS for y in range(int(l[:4]), int(l[5:]) + 1)}
ext_pool["period3"] = ext_pool["YEAR"].map(year_to_label3)
interact_labels3 = [b for b in EXT_BINS if b != REF3]
interact_cols3 = {b: "op3_" + b.replace("-", "_") for b in interact_labels3}
for b, c in interact_cols3.items():
    ext_pool[c] = ext_pool["outsourced"] * (ext_pool["period3"] == b)

# Linear trend term: unchanged
ext_pool["out_trend"] = ext_pool["outsourced"] * (ext_pool["YEAR"] - 2001)

def period_interaction_fit(s, interact_cols):
    names = list(interact_cols.values())
    f_int = author_formula.replace("outsourced + union", "outsourced + " + " + ".join(names) + " + union", 1)
    fit_int = pf.feols(f_int, data=s, weights="EARNWT", vcov={"CRV1": "ym"})
    coef_index = list(fit_int.coef().index)
    R = np.zeros((len(names), len(coef_index)))
    for i, n in enumerate(names):
        R[i, coef_index.index(n)] = 1
    wald = fit_int.wald_test(R=R, distribution="chi2")
    return fit_int, wald, names

rows_b = []
for occ in ["janitor", "guard"]:
    s = ext_pool[ext_pool["occupation"] == occ].dropna(subset=["log_wage", "union", "parttime", "educ_cat",
                  "female", "black", "hispanic", "age", "age2", "msa", "central_city", "cc_unknown", "geo_unidentified"]).copy()

    # Primary: 5-year period interactions + joint Wald
    fit5, wald5, names5 = period_interaction_fit(s, interact_cols5)
    tidy5 = fit5.tidy()
    for b, c in interact_cols5.items():
        r = tidy5.loc[c]
        rows_b.append(dict(occupation=OCC_LABEL[occ], sample="2001-2025",
                           test=f"Part B: outsourced x period {b} (relative to {REF5})",
                           estimate=r["Estimate"], se=r["Std. Error"], statistic=r["t value"], df=np.nan,
                           p=r["Pr(>|t|)"]))
    rows_b.append(dict(occupation=OCC_LABEL[occ], sample="2001-2025",
                       test="Part B: joint Wald, all outsourced x period interactions zero (chi2)",
                       estimate=np.nan, se=np.nan, statistic=float(wald5["statistic"]), df=len(names5),
                       p=float(wald5["pvalue"])))

    # Appendix robustness: 3-year period interactions + joint Wald
    fit3, wald3, names3 = period_interaction_fit(s, interact_cols3)
    tidy3 = fit3.tidy()
    for b, c in interact_cols3.items():
        r = tidy3.loc[c]
        rows_b.append(dict(occupation=OCC_LABEL[occ], sample="2001-2025 (3-year bins)",
                           test=f"Part B: outsourced x period {b} (relative to {REF3})",
                           estimate=r["Estimate"], se=r["Std. Error"], statistic=r["t value"], df=np.nan,
                           p=r["Pr(>|t|)"]))
    rows_b.append(dict(occupation=OCC_LABEL[occ], sample="2001-2025 (3-year bins)",
                       test="Part B: joint Wald, all outsourced x period interactions zero (chi2)",
                       estimate=np.nan, se=np.nan, statistic=float(wald3["statistic"]), df=len(names3),
                       p=float(wald3["pvalue"])))

    # Linear trend (unchanged formula/sample, independent of the period-interaction scheme)
    f_tr = author_formula.replace("outsourced + union", "outsourced + out_trend + union", 1)
    fit_tr = pf.feols(f_tr, data=s, weights="EARNWT", vcov={"CRV1": "ym"})
    r = fit_tr.tidy().loc["out_trend"]
    rows_b.append(dict(occupation=OCC_LABEL[occ], sample="2001-2025",
                       test="Part B: outsourced x linear year (centered 2001), per decade",
                       estimate=10 * r["Estimate"], se=10 * r["Std. Error"], statistic=r["t value"], df=np.nan,
                       p=r["Pr(>|t|)"]))

part_b = pd.DataFrame(rows_b)
print(part_b.round(4).to_string(index=False))

## Single years 1999–2005

Author's base model, one regression per year (from `02_estimates`). Plotted with 95% CIs; the dotted line sits between
2002 and 2003.

In [ ]:
SINGLE = [str(y) for y in range(1999, 2006)]
single = est[(est["spec"] == "author") & (est["model"] == "base") & (est["term"] == "outsourced")
             & (est["sample"].isin(SINGLE))].copy()
single["year"] = single["sample"].astype(int)
single_table = single[["occ", "year", "estimate", "se", "n_obs"]].sort_values(["occ", "year"]).reset_index(drop=True)
print(single_table.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {"janitor": "#404040", "guard": "#8c8c8c"}
markers = {"janitor": "o", "guard": "s"}
labels = {"janitor": "Janitors", "guard": "Guards"}
for occ in ["janitor", "guard"]:
    sub = single_table[single_table["occ"] == occ]
    ax.errorbar(sub["year"].to_numpy(), sub["estimate"].to_numpy(dtype=float),
                yerr=1.96 * sub["se"].to_numpy(dtype=float), marker=markers[occ], color=colors[occ],
                capsize=3, linewidth=1.5, label=labels[occ])
ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
ax.axvline(2002.5, color="black", linestyle=":", linewidth=1)
ax.set_xticks(range(1999, 2006))
ax.set_xlabel("Year")
ax.set_ylabel("Outsourced coefficient (log points)")
ax.set_title("Outsourcing Wage Penalty by Single Year, 1999-2005 (95% CI)")
ax.legend(title="Occupation")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUT}/figures/single_year_1999_2005.png", dpi=300)
plt.show()

In [ ]:
trend_tests = pd.concat([part_a, part_b], ignore_index=True)[
    ["occupation", "sample", "test", "estimate", "se", "statistic", "df", "p", "se_scaled", "p_scaled"]]
trend_tests.to_csv(f"{OUT}/py/trend_tests.csv", index=False)
print("saved outputs/py/trend_tests.csv", trend_tests.shape)

## Write RESULTS_trends.md

Descriptive summary of `outputs/py/trend_tests.csv` and the single-year estimates. Associations across periods, not causal effects.

In [ ]:
tt = trend_tests

def g(occ, samp, prefix):
    m = tt[(tt["occupation"] == occ) & (tt["sample"] == samp) & (tt["test"].str.startswith(prefix))]
    assert len(m) == 1, (occ, samp, prefix, len(m))
    return m.iloc[0]

def pfmt(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

def sgn(v, d=3):
    return f"{v:+.{d}f}"

def verdict(p):
    return "rejected" if p < 0.05 else "not rejected"

def trend_block(samp):
    out = []
    for occ in ["Janitors", "Guards"]:
        t = g(occ, samp, "Linear trend")
        out.append(f"| {occ} | {samp} | {sgn(t.estimate)} | {t.se:.3f} | {pfmt(t.p)} | {t.se_scaled:.3f} | {pfmt(t.p_scaled)} |")
    return out

def homog_block(samp):
    out = []
    for occ in ["Janitors", "Guards"]:
        h = g(occ, samp, "Homogeneity")
        out.append(f"- {occ}, {samp}: equal penalty across periods {verdict(h.p)} "
                   f"(Q = {h.statistic:.1f}, df = {int(h.df)}, {pfmt(h.p)}).")
    return out

def convergence_block(samp):
    h = g("Guard minus janitor", samp, "Convergence: homogeneity")
    t = g("Guard minus janitor", samp, "Convergence: linear trend")
    return [f"- {samp}: equal gap across periods {verdict(h.p)} (Q = {h.statistic:.1f}, df = {int(h.df)}, {pfmt(h.p)}). "
           f"Trend {sgn(t.estimate)} per decade (SE unscaled {t.se:.3f}, {pfmt(t.p)}; SE scaled {t.se_scaled:.3f}, "
           f"{pfmt(t.p_scaled)})."]

def contrast_lines(names):
    out = []
    for _, r in tt[tt["test"].isin(names)].iterrows():
        out.append(f"- {r['occupation']}, {r['test']}: difference {sgn(r['estimate'])} (SE {r['se']:.3f}), "
                   f"z = {r['statistic']:.2f}, {pfmt(r['p'])}.")
    return out

CONTRASTS_2001_2025 = ["Contrast: 2001-2003 minus 2021-2025 (change within extension)",
                       "Contrast: 2013-2015 minus 2021-2025 (guards only)"]
LONGRUN_CONTRASTS = ["Contrast: 1983-1985 minus 2021-2025 (long-run change)",
                     "Contrast: 1998-2000 minus 2001-2003 (replication/extension boundary)"]

trend_header = ["| Occupation | Sample | Trend per decade | SE (unscaled) | p | SE (scaled) | p (scaled) |",
               "|---|---|---|---|---|---|---|"]

# ---- Body: Part B is the primary trend test ----
lines = ["# Results: trend tests", "",
         "Regenerated by `notebooks/04_trends.ipynb` from `outputs/py/estimates.csv` and refit (Part B only). "
         "Associations across periods, not causal effects. Body is 2001-2025 (per CLAUDE.md's Paper structure); "
         "the appendix holds the 1983-2025 long-run series and robustness checks. Part B (pooled 2001-2025 model) "
         "is the primary trend test, with period interactions on the 5-year bins (2001-2005 .. 2021-2025, matching "
         "Table 3/Figure 2) and a joint Wald test that they are zero. Part A (by-period, meta-analytic, both "
         "3-year and 5-year bins), the pre-specified contrasts, the convergence test, and the 3-year-bin Part B "
         "robustness row are all in the appendix.", "",
         "## Body (2001-2025)", "",
         "### Part B: pooled model, linear trend and joint test (primary)", ""]
for occ in ["Janitors", "Guards"]:
    tb = g(occ, "2001-2025", "Part B: outsourced x linear")
    w5 = g(occ, "2001-2025", "Part B: joint Wald")
    lines.append(f"- {occ}: linear trend {sgn(tb.estimate)} per decade (SE {tb.se:.3f}, {pfmt(tb.p)}). Joint test "
                 f"that the 5-year period interactions are zero: {verdict(w5.p)} (chi2 = {w5.statistic:.1f}, "
                 f"df = {int(w5.df)}, {pfmt(w5.p)}).")
lines += ["", "The period interactions hold covariate coefficients constant across periods (unlike the by-period "
              "models in the appendix's Part A), so Part B can differ from Part A.", "",
          "### Single years, 1999-2005 (outsourced coefficient, SE)", ""]
piv = single_table.assign(cell=lambda d: d.apply(lambda r: f"{r['estimate']:+.3f} ({r['se']:.3f})", axis=1))
piv = piv.pivot(index="year", columns="occ", values="cell").reset_index()
lines += ["| Year | Janitors | Guards |", "|---|---|---|"]
for _, r in piv.iterrows():
    lines.append(f"| {int(r['year'])} | {r['janitor']} | {r['guard']} |")
lines += ["", "Plot: `outputs/figures/single_year_1999_2005.png`.", ""]

# ---- Appendix: Part A (3yr and 5yr), contrasts, convergence, 3-year Part B robustness, long-run series ----
lines += ["## Appendix (2001-2025 robustness, and 1983-2025 long-run)", "",
          "### Part B, 3-year-bin robustness (reference 2001-2003)", "",
          "Same pooled 2001-2025 model as the body's Part B, with the period interactions re-specified on the "
          "3-year/13-bin scheme used before this revision (Table A7/Figure C's bins). The linear trend is not "
          "repeated here -- it does not depend on the period-interaction scheme (see Body).", ""]
for occ in ["Janitors", "Guards"]:
    w3 = g(occ, "2001-2025 (3-year bins)", "Part B: joint Wald")
    lines.append(f"- {occ}: joint test that the 3-year period interactions are zero {verdict(w3.p)} "
                 f"(chi2 = {w3.statistic:.1f}, df = {int(w3.df)}, {pfmt(w3.p)}).")
lines += ["", "### Part A: by-period tests (meta-analytic, not model-based), 2001-2025", "",
          "3-year/13-bin series and 5-year-bin series (as in Table 3/Figure 2), shown as a bin-width check "
          "against Part B.", ""] + trend_header + trend_block("2001-2025") + trend_block("2001-2025 (5-year bins)") + [
          "", "Scaled SEs multiply the unscaled SE by sqrt(max(1, Q_resid / df_resid)).", "",
          "#### Homogeneity across periods", ""] + homog_block("2001-2025") + homog_block("2001-2025 (5-year bins)") + [
          "", "### Pre-specified contrasts, within 2001-2025 (unscaled SE)", ""] + contrast_lines(CONTRASTS_2001_2025) + [
          "", "### Convergence: guard minus janitor gap, 2001-2025", ""] + convergence_block("2001-2025") + [
          "", "### Long-run series, 1983-2025", "",
          "#### Linear trend per decade (log points)", ""] + trend_header + trend_block("1983-2025") + [
          "", "#### Homogeneity across periods", ""] + homog_block("1983-2025") + [
          "", "#### Long-run contrasts (unscaled SE)", ""] + contrast_lines(LONGRUN_CONTRASTS) + [
          "", "#### Convergence: guard minus janitor gap, 1983-2025", ""] + convergence_block("1983-2025") + [""]

with open(f"{OUT}/RESULTS_trends.md", "w") as f:
    f.write("\n".join(lines))
print("\n".join(lines))